In [ ]:
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import rc
matplotlib.rcParams['figure.dpi'] = 500
matplotlib.rcParams['text.usetex'] = True
rc("animation", html = "jshtml")
from matplotlib.colors import ListedColormap
# plt.rcParams['font.family'] = 'serif'
# plt.rcParams['font.serif'] = ['Times New Roman', 'Palatino', 'Computer Modern Roman']
plt.rcParams['font.size'] = 16
# plt.style.use('default')
plt.style.use('./data/plots/desi.mplstyle')

import numpy as np
from sklearn.datasets import make_blobs
import umap
from sklearn.neighbors import radius_neighbors_graph
from scipy.sparse.csgraph import connected_components
import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.patches import Circle
import pandas as pd

from collections import defaultdict
from astropy.convolution import Gaussian1DKernel
from astropy.convolution import convolve
import h5py

import os
os.environ['PATH'] = '/Library/TeX/texbin:' + os.environ['PATH']
from pathlib import Path

# umap

In [ ]:
cmap = sns.color_palette('mako', as_cmap=True)
cmap

In [ ]:
cmap(0.4)

In [ ]:
col = cmap(np.linspace(0., 1., 10))
rgb_floats = col[:, :3]

rgb_ints = (col[:, :3] * 255).astype(int)

# convierte cada tripleta (R,G,B) a cadena hexadecimal
hex_codes = ['#{:02x}{:02x}{:02x}'.format(r, g, b) for r, g, b in rgb_ints]

print(hex_codes)

In [ ]:
X, _ = make_blobs(n_samples=600, centers=4, cluster_std=9.)

reducer = umap.UMAP(
    n_neighbors=100,
    min_dist=0.0,
    spread=0.27
)
X_emb = reducer.fit_transform(X)

radius = 0.1
adj = radius_neighbors_graph(X_emb, radius=radius, include_self=False)

n_components, labels = connected_components(adj, directed=False)

rows, cols = adj.nonzero()
G = nx.Graph()
G.add_edges_from(zip(rows, cols))

pos = {i: X_emb[i] for i in range(len(X_emb))}
cmap = ListedColormap(sns.color_palette("mako", n_components))

plt.figure()
nx.draw_networkx_edges(G, pos, alpha=0.3, width=0.5)

for cluster_id in range(n_components):
    nodelist = np.where(labels == cluster_id)[0]
    nx.draw_networkx_nodes(
        G, pos,
        nodelist=nodelist,
        node_size=60,
        node_color=[cmap(cluster_id)],
        label=f'Cluster {cluster_id+1}',
        alpha=0.8
    )

plt.axis('off')
# plt.legend(markerscale=1, fontsize='small', loc='upper right', title='Clusters')
# plt.title('UMAP + FoF como Grafo con colores únicos por cluster')
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
from sklearn.datasets import make_blobs
import umap
from sklearn.neighbors import radius_neighbors_graph
from scipy.sparse.csgraph import connected_components
import matplotlib.pyplot as plt
import networkx as nx
import seaborn as sns
from matplotlib.colors import ListedColormap

# 1) Datos más separados (clusters más compactos alrededor de centros alejados)
X, _ = make_blobs(
    n_samples=600,
    centers=4,
    cluster_std=4.0,      # reduce la dispersión interna
    random_state=42
)

# 2) UMAP con mayor separación entre grupos
reducer = umap.UMAP(
    n_neighbors=50,      # menos vecinos → grupos más independientes
    min_dist=0.3,        # distancia mínima >0 → evita que se junten demasiado
    spread=1.0,          # mayor “spread” → más distancia global
    random_state=42
)
X_emb = reducer.fit_transform(X)

# 3) Grafo FoF con un radius acorde a la escala de X_emb
radius = 0.5
adj = radius_neighbors_graph(X_emb, radius=radius, include_self=False)

# 4) Etiquetas FoF
n_components, labels = connected_components(adj, directed=False)

# 5) Construcción del grafo
rows, cols = adj.nonzero()
G = nx.Graph()
G.add_edges_from(zip(rows, cols))

# 6) Posiciones y colormap
pos = {i: X_emb[i] for i in range(len(X_emb))}
cmap = ListedColormap(sns.color_palette("mako", n_components))

# 7) Dibujo
fig, ax = plt.subplots(figsize=(10, 8))

# — aristas
nx.draw_networkx_edges(G, pos, alpha=0.3, width=0.5)

# — nodos por cluster
for cid in range(n_components):
    nodelist = np.where(labels == cid)[0]
    nx.draw_networkx_nodes(
        G, pos,
        nodelist=nodelist,
        node_size=60,
        node_color=[cmap(cid)],
        label=f'Cluster {cid+1}',
        alpha=0.8
    )

# aspecto igual para que todo respete la escala
ax.set_aspect('equal', 'box')
ax.axis('off')
plt.tight_layout()
plt.show()


## 1

In [ ]:
y_true = _

In [ ]:
num_clusters = len(np.unique(y_true))
cmap = ListedColormap(sns.color_palette("mako", num_clusters))

fig, ax = plt.subplots()
scatter = ax.scatter(
    X_emb[:, 0],
    X_emb[:, 1],
    c=y_true,
    cmap=cmap,
    s=60,
    alpha=0.8
)

plt.axis('off')

# Construir leyenda a partir del scatter
handles_, labels_ = scatter.legend_elements(prop="colors", alpha=0.8)
labels_ = ['Category A', 'Category B', 'Category C']
# labels vienen como ['0', '1', '2', ...], los convertimos a 'Cluster 0', etc.
labels_ = [lbl for lbl in labels_]
ax.legend(handles_, labels_, loc='upper right', 
           markerscale=2, fontsize=18, title_fontsize=22,
           framealpha=1)

ax.set_aspect('equal', 'box')   # hace iguales las unidades X e Y
# ax.relim()                      # recalcula los límites con el nuevo aspecto
# ax.autoscale_view()
plt.tight_layout()
plt.show()

## 2

In [ ]:
y_true = _
centers = np.array([X_emb[labels == i].mean(axis=0) for i in range(n_components)])

# — Determinar etiqueta verdadera dominante en cada FoF-cluster
dominant_labels = []
for i in range(n_components):
    idx = np.where(labels == i)[0]
    counts = np.bincount(y_true[idx])
    dominant_labels.append(np.argmax(counts))
dominant_labels = np.array(dominant_labels)

num_true = len(np.unique(y_true))
cmap_true = ListedColormap(sns.color_palette("mako", num_true))

fig, ax = plt.subplots()
radius = 1.2
for i, center in enumerate(centers):
    circ = Circle(
        center,
        radius,
        edgecolor=cmap_true(dominant_labels[i]),
        facecolor='none',
        linestyle='--',
        linewidth=1.4
    )
    ax.add_patch(circ)

# 2) Scatter de puntos coloreados por etiqueta verdadera
for lbl in np.unique(y_true):
    pts = np.where(y_true == lbl)[0]
    ax.scatter(
        X_emb[pts, 0],
        X_emb[pts, 1],
        s=60,
        c=[cmap_true(lbl)],
        label=f'Cluster {lbl}',
        alpha=0.8
    )

center0 = centers[4]
start0 = np.array([radius, 0]) +center0
end0 = center0

ax.plot(
    [start0[0], end0[0]],
    [start0[1], end0[1]],
    color=cmap(0.2),  # o el color que uses
    linewidth=1.7
)

cap = radius * 0.08
ax.plot(
    [start0[0], start0[0]],
    [start0[1] - cap, start0[1] + cap],
    color='k',
    linewidth=1.
)
ax.plot(
    [end0[0], end0[0]],
    [end0[1] - cap, end0[1] + cap],
    color='k',
    linewidth=1.
)

mid0 = (start0 + end0) / 2
ax.text(
    mid0[0],
    mid0[1],
    r'$r$',
    fontsize=24,
    color=cmap(0.2),
    va='bottom',
    ha='center'
)
ax.set_aspect('equal', 'box')   # hace iguales las unidades X e Y
# ax.relim()                      # recalcula los límites con el nuevo aspecto
# ax.autoscale_view()

ax.axis('off')
plt.tight_layout()
plt.show()


## 3

In [ ]:
num_true = len(np.unique(y_true))
cmap_true = ListedColormap(sns.color_palette("mako", num_true))

# 5) Dibujado
fig, ax = plt.subplots()

# 5.1) Aristas
nx.draw_networkx_edges(G, pos, alpha=0.9, width=0.6)

# 5.2) Nodos, filtrando por los que sí están en G
for lbl in np.unique(y_true):
    pts = np.where(y_true == lbl)[0]
    ax.scatter(
        X_emb[pts, 0],
        X_emb[pts, 1],
        s=60,
        c=[cmap_true(lbl)],
        label=f'Cluster {lbl}',
        alpha=0.7
    )

#----------------

# plt.title('Grafo FoF con colores de etiqueta original')

ax.set_aspect('equal', 'box')   # hace iguales las unidades X e Y
# ax.relim()                      # recalcula los límites con el nuevo aspecto
# ax.autoscale_view()
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.colors import ListedColormap
import seaborn as sns
import networkx as nx
from sklearn.datasets import make_blobs
import umap
from sklearn.neighbors import radius_neighbors_graph
from scipy.sparse.csgraph import connected_components

# 1) Generar datos de ejemplo
X, _ = make_blobs(
    n_samples=600,
    centers=4,
    cluster_std=9.0,
)

# 2) UMAP para embeder con separación
reducer = umap.UMAP(
    n_neighbors=50,
    min_dist=0.3,
    spread=1.0,
)
X_emb = reducer.fit_transform(X)

# 3) Grafo FoF
radius = 0.45
adj = radius_neighbors_graph(X_emb, radius=radius, include_self=False)

# 4) Etiquetas FoF
n_components, labels = connected_components(adj, directed=False)

# 5) Construir grafo y posiciones
rows, cols = adj.nonzero()
G = nx.Graph()
G.add_edges_from(zip(rows, cols))
# pos para networkx
pos = {i: X_emb[i] for i in range(len(X_emb))}

# 6) Calcular tamaños de cluster y filtrar ≤2
unique_labels, counts = np.unique(labels, return_counts=True)
cluster_sizes = dict(zip(unique_labels, counts))
small_labels = [lbl for lbl, cnt in cluster_sizes.items() if cnt <= 2]
print("Tamaños de cluster:", cluster_sizes)
print("Clusters pequeños (<=2):", small_labels)

# 7) Preparar colormap
cmap = ListedColormap(sns.color_palette("mako", len(np.unique(_))))

# 8) Dibujar
fig, ax = plt.subplots()
nx.draw_networkx_edges(G, pos, alpha=0.7, width=0.8)

# scatter de todos los puntos
# scatter = ax.scatter(
#     X_emb[:, 0], X_emb[:, 1],
#     c=labels, cmap=cmap,
#     s=60, alpha=0.7, zorder=1
# )
y_true = _
for lbl in np.unique(y_true):
    pts = np.where(y_true == lbl)[0]
    ax.scatter(
        X_emb[pts, 0],
        X_emb[pts, 1],
        s=60,
        c=[cmap(lbl)],
        label=f'Cluster {lbl}',
        alpha=0.7
    )

# 9) Añadir círculos a clusters pequeños
for lbl in small_labels:
    idx = np.where(labels == lbl)[0]
    coords = X_emb[idx]
    x_c, y_c = coords.mean(axis=0)
    if len(coords) > 1:
        dists = np.linalg.norm(coords - [x_c, y_c], axis=1)
        radius_c = dists.max() + 0.5
    else:
        radius_c = 0.6

    circ = Circle(
        (x_c, y_c),
        radius=radius_c,
        edgecolor='red',
        facecolor='none',
        linewidth=1.5,
        linestyle='--',
        zorder=2
    )
    ax.add_patch(circ)

# 10) Ajustar límites y estilo
# ax.relim()
# ax.autoscale_view()
ax.set_aspect('equal', 'box')
ax.axis('off')
# plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots()

y_true = _
for lbl in np.unique(y_true):
    pts = np.where(y_true == lbl)[0]
    ax.scatter(
        X_emb[pts, 0],
        X_emb[pts, 1],
        s=60,
        c=[cmap(lbl)],
        label=f'Cluster {lbl}',
        alpha=0.7
    )

ax.set_aspect('equal', 'box')
ax.axis('off')
# plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots()

nx.draw_networkx_edges(G, pos, alpha=0.5, width=0.6)
y_true = _
for lbl in np.unique(y_true):
    pts = np.where(y_true == lbl)[0]
    ax.scatter(
        X_emb[pts, 0],
        X_emb[pts, 1],
        s=5,
        c=[cmap(lbl)],
        label=f'Cluster {lbl}',
        alpha=0.7
    )

ax.set_aspect('equal', 'box')
ax.axis('off')
# plt.tight_layout()
plt.show()


In [ ]:
# 3) Grafo FoF
radius = 0.45
adj = radius_neighbors_graph(X_emb, radius=radius, include_self=False)

# 4) Etiquetas FoF
n_components, labels = connected_components(adj, directed=False)

# 5) Construir grafo y posiciones
rows, cols = adj.nonzero()
G = nx.Graph()
G.add_edges_from(zip(rows, cols))
pos = {i: X_emb[i] for i in range(len(X_emb))}

# 6) Calcular clusters pequeños
unique_labels, counts = np.unique(labels, return_counts=True)
cluster_sizes = dict(zip(unique_labels, counts))
small_labels = [lbl for lbl, cnt in cluster_sizes.items() if cnt <= 2]

# 7) Preparar colormap para los 4 clusters
n_clusters_true = len(np.unique(y_true))
cmap = ListedColormap(sns.color_palette("mako", n_clusters_true))

# 8) Crear figura con 3 subplots
fig, (ax0, ax1, ax2) = plt.subplots(1, 3, figsize=(15, 5))

# — Panel 1: sólo el embedding coloreado por y_true —
for lbl in np.unique(y_true):
    pts = np.where(y_true == lbl)[0]
    ax0.scatter(
        X_emb[pts, 0], X_emb[pts, 1],
        s=50, c=[cmap(lbl)], alpha=0.7,
        label=f'Cluster {lbl}'
    )
ax0.set_title("UMAP embedding", y=1.02)
ax0.axis("off")
ax0.set_aspect('equal')

# — Panel 2: el grafo FoF (aristas + nodos) —
# dibujar sólo aristas
nx.draw_networkx_edges(
    G, pos,
    alpha=0.7, width=0.6,
    ax=ax1
)
# luego dibujar nodos encima
ax1.scatter(
    X_emb[:, 0], X_emb[:, 1],
    s=5,
    c='black',
    alpha=0.7
)
ax1.set_title("FoF graph", y=1.02)
ax1.axis("off")
ax1.set_aspect('equal')

# — Panel 3: embedding + círculos en clusters pequeños —
nx.draw_networkx_edges(
    G, pos,
    alpha=0.9, width=0.6,
    ax=ax2
)

for lbl in np.unique(y_true):
    pts = np.where(y_true == lbl)[0]
    ax2.scatter(
        X_emb[pts, 0], X_emb[pts, 1],
        s=50, c=[cmap(lbl)], alpha=0.7
    )
# dibujar círculos alrededor de clusters de tamaño ≤2
for lbl in small_labels:
    idx = np.where(labels == lbl)[0]
    coords = X_emb[idx]
    x_c, y_c = coords.mean(axis=0)
    if len(coords) > 1:
        dists = np.linalg.norm(coords - [x_c, y_c], axis=1)
        radius_c = dists.max() + 0.5
    else:
        radius_c = 0.6
    circ = Circle(
        (x_c, y_c), radius=radius_c,
        edgecolor='red', facecolor='none',
        linewidth=1.5, linestyle='--', zorder=2
    )
    ax2.add_patch(circ)

ax2.set_title("Outliers identified", y=1.02)
ax2.axis("off")
ax2.set_aspect('equal')

plt.tight_layout()
plt.show()

# spectra

In [ ]:
file = np.load('./data/processed/umap/umap_20211110_10256.npz', allow_pickle=True)

In [ ]:
file.files

In [ ]:
def load_data(npz_path):
    data = np.load(npz_path, allow_pickle=True)
    embedding = data['embedding']
    raw_cats = data['categories']

    categories = [c.decode('utf-8') if isinstance(c, (bytes, bytearray)) else str(c)
                  for c in raw_cats]

    outlier_mask = data['outlier_mask']
    df = pd.DataFrame({
        'UMAP1': embedding[:, 0],
        'UMAP2': embedding[:, 1],
        'category': categories,
        'is_outlier': outlier_mask
        })
    df.meta_n_clusters = len(np.unique(data['labels']))
    return df

In [ ]:
def plot_umap(df, tile_id, night, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(9,8))

    cats = sorted(df['category'].unique())
    palette = sns.color_palette('mako_r', n_colors=len(cats))

    color_dict = dict(zip(cats, palette))

    for c in cats:
        mask = (df['category'] == c) & (~df['is_outlier'])
        ax.scatter(
            df.loc[mask, 'UMAP1'],
            df.loc[mask, 'UMAP2'],
            s=20,
            color=color_dict[c],
            label=c,
            alpha=1,
            edgecolor='black',
            linewidth=0.1
        )

    o = df['is_outlier']
    if o.any():
        ax.scatter(
            df.loc[o, 'UMAP1'],
            df.loc[o, 'UMAP2'],
            s=50,
            marker='x',
            linewidths=1.5,
            color='black',
            label='Outliers'
        )

    ax.legend()
    ax.set_xticks([]); ax.set_yticks([])
    plt.axis('off')

    title = (f'\n{len(df)} spectra, {df.meta_n_clusters} clusters\n'
             fr'{o.sum()} outliers, {o.sum()/len(df)*100:.2f}\%')
    fig.suptitle(f'{night} - {tile_id}\n', fontsize=21, weight='bold', y=0.99)
    ax.set_title(title, y=0.98, fontsize=18)
    ax.set_aspect('equal', 'box')

    plt.show()

In [ ]:
data = load_data('./data/processed/umap/umap_20211110_10256.npz')

In [ ]:
link_length = 0.45
graph = radius_neighbors_graph(emb, radius=link_length,
                                       mode='connectivity', include_self=True,
                                       n_jobs=-1)
n_clusters, labels = connected_components(csgraph=graph,
                                                            directed=False,
                                                            return_labels=True)

min_cluster_size = 5
uniq, cnt = np.unique(labels, return_counts=True)
small = uniq[cnt <= min_cluster_size]
a = np.isin(labels, small)

In [ ]:
plot_umap(data, '10256', '20211110', './data/processed/umap/plots')


In [ ]:
c: cmap(i / (len(cats)-1) * 0.7 + 0.1) for i,c in enumerate(cats)}

# Otra

In [ ]:
import sys, os
project_root = os.path.abspath('..')
sys.path.insert(0, project_root)

from src.desiproc.build_matrix import build_matrix
import glob

out_dir = os.path.join(project_root, 'AssessingDesiData', 'data', 'processed')
night   = '20211130'
tile    = '5568'
bands   = ['B','R','Z']
wg, fp, iv, z, ze, ids, cat, petals = build_matrix(out_dir, night, tile, bands)

print("wave_grid:", wg.shape)
print("flux matrix:", fp.shape)
print("ivar matrix:", iv.shape)
print("z vector:", z.shape)
print("zerr vector:", ze.shape)
print("ids vector:", ids.shape)
print("cat matrix:", cat.shape)
print("petals matrix:", petals.shape)

In [ ]:
X, _ = fp, cat
y_true = _

In [ ]:
reducer = umap.UMAP(
    n_neighbors=50,      # menos vecinos → grupos más independientes
    min_dist=1.0,        # distancia mínima >0 → evita que se junten demasiado
    # spread=1.0,          # mayor “spread” → más distancia global
    # random_state=42
)
X_emb = reducer.fit_transform(X)
plt.plot(X_emb[:, 0], X_emb[:, 1], 'o', markersize=2, alpha=0.5)

In [ ]:

# 3) Grafo FoF con un radius acorde a la escala de X_emb
radius = 0.1
adj = radius_neighbors_graph(X_emb, radius=radius, include_self=False)

# 4) Etiquetas FoF
n_components, labels = connected_components(adj, directed=False)

# 5) Construcción del grafo
rows, cols = adj.nonzero()
G = nx.Graph()
G.add_edges_from(zip(rows, cols))

# 6) Posiciones y colormap
pos = {i: X_emb[i] for i in range(len(X_emb))}

In [ ]:
num_true = len(np.unique(y_true))
cmap_true = ListedColormap(sns.color_palette("mako", num_true))

# 5) Dibujado
fig, ax = plt.subplots()

# 5.1) Aristas
nx.draw_networkx_edges(G, pos, alpha=0.7, width=0.5)

# 5.2) Nodos, filtrando por los que sí están en G
node_list = list(G.nodes())
node_colors = y_true[node_list]    # <— sólo etiquetas de esos índices
nx.draw_networkx_nodes(
    G, pos,
    nodelist=node_list,
    node_size=40,
    # node_color=node_colors,
    # cmap=cmap_true,
    alpha=0.8
)

# plt.title('Grafo FoF con colores de etiqueta original')

ax.set_aspect('equal', 'box')   # hace iguales las unidades X e Y
ax.relim()                      # recalcula los límites con el nuevo aspecto
ax.autoscale_view()
plt.axis('off')
plt.tight_layout()
plt.show()

----

In [ ]:
def plot_outlier_spectra(npz_file, out_dir, night, tile, plot_path=None):
    data = np.load(npz_file, allow_pickle=True)
    mask = data['outlier_mask']
    ids_all = data['ids'][mask].astype(int)
    petals = data['petals'][mask].astype(int)
    types_all = data['categories'][mask]
    types_all = [c.decode('utf-8') if isinstance(c, (bytes, bytearray)) else str(c)
                  for c in types_all]
    type_map = dict(zip(ids_all, types_all))

    petal_groups = defaultdict(list)
    for tgt_id, petal in zip(ids_all, petals):
        petal_groups[petal].append(tgt_id)

    # if plot_path is None:
    #     plot_path = os.path.join('../data', 'plots', 'spectra', night)
    # os.makedirs(plot_path, exist_ok=True)

    kernel = Gaussian1DKernel(5)

    for petal, tgt_list in petal_groups.items():
        h5_fn = os.path.join(out_dir, f'{night}-{tile}-{petal}.h5')

        with h5py.File(h5_fn, 'r') as f:
            all_ids = f['metadata/target_id'][:].astype(int)
            idxs = np.nonzero(np.in1d(all_ids, tgt_list))[0]

            waves, fluxes, smooth = {}, {}, {}
            for band in ('B','R','Z'):
                grp = f[f'spectra/{band}']
                w = grp['wavelength'][:]
                fl = grp['flux'][idxs, :]
                sf = np.vstack([convolve(row, kernel) for row in fl])
                waves[band]   = w
                fluxes[band]  = fl
                smooth[band]  = sf

        for i, tgt_id in enumerate([all_ids[j] for j in idxs[:2]]):
            plt.figure(figsize=(18, 6))
            for band, clr in zip(('B','R','Z'), ('b','g','r')):
                plt.plot(waves[band], fluxes[band][i], color=clr, alpha=0.5)
                plt.plot(waves[band], smooth[band][i], color='k', linewidth=0.8)

            plt.rcParams['font.size'] = 18

            plt.xlim(3500, 9900)
            plt.xlabel(r'$\lambda$ [$\mathrm{\AA}$]')
            plt.ylabel(r'$F_{\lambda}$ [$10^{-17}\ \mathrm{erg}\,\mathrm{s}^{-1}\,\mathrm{cm}^{-2}\,\mathrm{\AA}^{-1}$]')
            plt.grid(linewidth=0.5)
            # plt.title(f"{type_map[tgt_id]} - ID: {tgt_id}\nNight: {night}, Tile: {tile}, Petal: {petal}", y=1.05)
            plt.tight_layout()
            plt.show()

            # out_png = os.path.join(plot_path, f'spec_{tile}_{petal}_{tgt_id}.png')
            # plt.savefig(out_png, dpi=200)
            # plt.close()

In [ ]:
plot_outlier_spectra('./data/processed/umap/umap_20211110_10256.npz',
                     out_dir='./data/processed',
                     night='20211110',
                     tile='10256',)

In [ ]:
def plot_outlier_spectra(npz_file, out_dir, night, tile, plot_path=None):
    data = np.load(npz_file, allow_pickle=True)
    mask = data['outlier_mask']
    ids_all = data['ids'][mask].astype(int)
    petals = data['petals'][mask].astype(int)
    types_all = data['categories'][mask]
    types_all = [c.decode('utf-8') if isinstance(c, (bytes, bytearray)) else str(c)
                 for c in types_all]
    type_map = dict(zip(ids_all, types_all))

    # group by petal
    petal_groups = defaultdict(list)
    for tgt_id, petal in zip(ids_all, petals):
        petal_groups[petal].append(tgt_id)

    # define lines to annotate (rest wavelengths in Å)
    # emission_lines = {
    #     '$\text{H}\beta': 4861.33,
    #     '$\left[\text{O} III\right]': 5006.84,
    #     '\text{H}\alpha': 6562.80,
    #     '$\text{Ca} K$': 3933.66,
    #     '$\text{Ca} H$': 3968.47
    # }
    emission_lines = {
        r'$\mathrm{H}\beta$': 4861.33,
        r'$\mathrm{O} 3$': 5006.84,
        r'$\mathrm{H} \alpha$': 6562.80,
        r'$\mathrm{Ca} \mathrm{K}$': 3933.66,
        # r'$\mathrm{Ca} \mathrm{H}$': 3968.47
    }

    kernel = Gaussian1DKernel(5)

    for petal, tgt_list in petal_groups.items():
        print(f'Petal {petal} ({len(tgt_list)} targets)')
        h5_fn = os.path.join(out_dir, f'{night}-{tile}-{petal}.h5')
        with h5py.File(h5_fn, 'r') as f:
            all_ids = f['metadata/target_id'][:].astype(int)
            idxs = np.nonzero(np.in1d(all_ids, tgt_list))[0]

            # read and smooth each band
            waves, fluxes, smooth = {}, {}, {}
            for band in ('B','R','Z'):
                grp = f[f'spectra/{band}']
                w = grp['wavelength'][:]
                fl = grp['flux'][idxs, :]
                sf = np.vstack([convolve(row, kernel) for row in fl])
                waves[band], fluxes[band], smooth[band] = w, fl, sf

        # now plot each outlier spectrum
        for i, tgt_id in enumerate(all_ids[idxs]):
            plt.figure(figsize=(18, 6))
            # raw + smoothed
            for band, clr in zip(('B','R','Z'), ('b','g','r')):
                plt.plot(waves[band], fluxes[band][i], color=clr, alpha=0.5)
                plt.plot(waves[band], smooth[band][i], color='k', linewidth=0.8)

            # annotate emission/absorption lines
            ymin, ymax = plt.ylim()
            for name, lam in emission_lines.items():
                if 3500 < lam < 9900:
                    plt.axvline(lam, linestyle='--', linewidth=0.5, color='k')
                    plt.text(
                        lam, ymax*0.7, name,
                        rotation=0, va='top', ha='right',
                        fontsize=16, backgroundcolor='white'
                    )

            plt.xlim(3500, 9900)
            plt.rcParams['font.size'] = 16
            plt.xticks(fontsize=16)
            

            plt.xlabel(r'$\lambda$ [$\mathrm{\AA}$]')
            plt.ylabel(r'$F_{\lambda}$ [$10^{-17}\ \mathrm{erg}\,\mathrm{s}^{-1}\,\mathrm{cm}^{-2}\,\mathrm{\AA}^{-1}$]')
            plt.grid(linewidth=0.2)
            plt.title(
                f"{type_map[tgt_id]} – ID: {tgt_id}\n"
                f"Night: {night}, Tile: {tile}, Petal: {petal}",
                y=1.05
            )
            plt.tight_layout()
            plt.show()
            

In [ ]:
plot_outlier_spectra('./data/processed/umap/umap_20211110_10256.npz',
                     out_dir='./data/processed',
                     night='20211110',
                     tile='10256',)

In [ ]:
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from astropy.convolution import convolve, Gaussian1DKernel

# parameters
night, tile, petal = "20211110", "10256", 8
target_id = 39627563059908008
data_dir = "./data/processed"   # e.g. "../data/processed"
h5path = os.path.join(data_dir, f"{night}-{tile}-{petal}.h5")

# emission / absorption windows to zoom on: (center, half-width, label)
windows = [
    (3727.0, 30, "[O II]"),
    (4861.3, 30, "Hbeta"),
    (5007.0, 30, "[O III]"),
    (5892.0, 30, "Na D"),
    (6562.8, 30, "Halpha"),
]

# colors for bands
band_colors = dict(B="b", R="g", Z="r")

# load the spectrum
with h5py.File(h5path, "r") as f:
    ids = f["metadata/target_id"][:].astype(int)
    i = np.where(ids == target_id)[0]
    if len(i)==0:
        raise ValueError(f"{target_id} not in file")
    i = int(i[0])
    waves, fluxes, smooth = {}, {}, {}
    kernel = Gaussian1DKernel(5)
    for band in ("B","R","Z"):
        grp = f[f"spectra/{band}"]
        w = grp["wavelength"][:]
        fl = grp["flux"][i]
        sf = convolve(fl, kernel)
        waves[band], fluxes[band], smooth[band] = w, fl, sf

# set up 1×(1+len(windows)) panels
nzoom = len(windows)
fig = plt.figure(figsize=(18, 6))
gs = GridSpec(1, nzoom+1, width_ratios=[3]+[1]*nzoom, wspace=0.3)

# panel 0: full 3500–9900
ax0 = fig.add_subplot(gs[0,0])
for band in ("B","R","Z"):
    ax0.plot(waves[band], fluxes[band], color=band_colors[band], alpha=0.5)
    ax0.plot(waves[band], smooth[band], color="k", lw=0.8)
ax0.set_xlim(3500,9900)
ax0.set_ylim(0, np.max([smooth[b].max() for b in ("B","R","Z")])*1.1)
ax0.set_ylabel("Normalized flux")
ax0.set_title(f"Full band  {night}  tile {tile} petal {petal}")
ax0.grid(False)

# panels 1…n: zooms
for j,(center,hw,label) in enumerate(windows, start=1):
    ax = fig.add_subplot(gs[0,j])
    xmin,xmax = center-hw, center+hw
    for band in ("B","R","Z"):
        mask = (waves[band]>=xmin)&(waves[band]<=xmax)
        ax.plot(waves[band][mask], fluxes[band][mask], color=band_colors[band], alpha=0.5)
        ax.plot(waves[band][mask], smooth[band][mask], color="k", lw=0.8)
    ax.axvline(center, ls="--", color="k")
    ax.set_xlim(xmin,xmax)
    ax.set_title(label, pad=1)
    ax.set_xticks([])
    ax.set_yticks([])

# plt.tight_layout()
plt.show()


In [ ]:
import os
import numpy as np
import h5py
from sklearn.neighbors import radius_neighbors_graph
from scipy.sparse.csgraph import connected_components
import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.cm import ScalarMappable

# parameters
npz_file = "./data/processed/umap/umap_20211110_10256.npz"   # contains X_emb, ids, petals
out_dir   = "./data/processed"   # e.g. "../data/processed"
night, tile = "20211110", "10256"
radius    = 0.3

# 1) load embedding + IDs
data    = np.load(npz_file, allow_pickle=True)
X_emb   = data["embedding"]            # shape (N,2)
ids_all = data["ids"].astype(int)      # shape (N,)
petals  = data["petals"].astype(int)   # shape (N,)
outl = data["outlier_mask"]          # shape (N,)

# 2) for each (id,petal) find its redshift in the HDF5
#    build a map id -> z  
z_map = {}
for petal in np.unique(petals):
    fn = os.path.join(out_dir, f"{night}-{tile}-{petal}.h5")
    with h5py.File(fn,"r") as f:
        tgt_ids = f["metadata/target_id"][:].astype(int)
        zs      = f["metadata/redrock_z"][:]      # adjust path if different
        for tid, z in zip(tgt_ids, zs):
            z_map[tid] = z

In [ ]:
np.sum(outl)

In [ ]:
# 3) collect redshift array aligned with X_emb
z_vals = np.array([z_map[tid] for tid in ids_all])

# 4) build FoF adjacency
adj = radius_neighbors_graph(X_emb, radius=radius, include_self=False)

# 5) build NetworkX graph
rows, cols = adj.nonzero()
G = nx.Graph()
G.add_edges_from(zip(rows, cols))

# 6) plot colored by redshift
plt.figure(figsize=(8,6))
pos = {i: X_emb[i] for i in range(len(X_emb))}

# edges
nx.draw_networkx_edges(G, pos, alpha=0.2, width=0.5)

# nodes
nodes = nx.draw_networkx_nodes(
    G, pos,
    node_size=5,
    node_color=z_vals[list(G.nodes())],
    cmap=sns.color_palette("mako", as_cmap=True),
    vmin=z_vals.min(),
    vmax=z_vals.max()
)

# plt.colorbar(ScalarMappable(cmap=sns.color_palette("mako", as_cmap=True), 
#                             norm=plt.Normalize(vmin=z_vals.min(), vmax=z_vals.max())),
#              label="z$$")
cbar = fig.colorbar(nodes, ax=ax, label="Redshift")


plt.title("UMAP + FoF graph, colored by redshift")
plt.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
outliers = np.where(outl)[0]

min_size, max_size = 20, 200
zmin, zmax = z_vals.min(), z_vals.max()
sizes = (min_size + (z_vals - zmin) / (zmax - zmin) * (max_size - min_size)) * 0.01

pos = {i: X_emb[i] for i in range(len(X_emb))}

fig, ax = plt.subplots(figsize=(8, 6))

adj = radius_neighbors_graph(X_emb, radius=0.4, include_self=False)

# 5) build NetworkX grap
nx.draw_networkx_edges(G, pos, alpha=0.2, width=0.5, ax=ax)

nx.draw_networkx_nodes(
    G, pos,
    nodelist=list(G.nodes()),
    node_size=1,
    node_color='k',
    alpha=0.8,
    ax=ax
)

nx.draw_networkx_nodes(
    G, pos,
    nodelist=outliers,
    node_size=5,
    node_color='red',
    alpha=0.9,
    ax=ax
)

ax.set_title("UMAP + FoF graph")
ax.set_aspect('equal', 'box')
ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# … (load X_emb, ids_all, z_vals, build adj & G as before) …
out = X_emb[outl, :].shape
# choose min/max marker sizes
min_size, max_size = 20, 200
zmin, zmax = z_vals.min(), z_vals.max()
# linear mapping: size = min_size + (z - zmin)/(zmax-zmin)*(max_size-min_size)
sizes = (min_size + (z_vals - zmin) / (zmax - zmin) * (max_size - min_size))*0.01

z_examples = [zmin, 0.5*(zmin+zmax), zmax]
size_examples = (min_size + (np.array(z_examples)-zmin)/(zmax-zmin)*(max_size-min_size))

# plot
fig, ax = plt.subplots(figsize=(8, 6))
pos = {i: X_emb[i] for i in range(len(X_emb))}

# draw edges
nx.draw_networkx_edges(G, pos, alpha=0.2, width=0.5, ax=ax)

# draw nodes with size by redshift
nx.draw_networkx_nodes(
    G, pos,
    nodelist=list(G.nodes()),
    node_size=sizes[list(G.nodes())],
    node_color='k',        # or keep a fixed color
    alpha=0.8,
    ax=ax
)

handles = []
for z_val, sz in zip(z_examples, size_examples):
    handles.append(
        ax.scatter([], [], s=sz*0.1, color='k', alpha=1,
                   label=f"z = {z_val:.3f}")
    )
leg = ax.legend(handles=handles, title="$z$ → marker size",
                scatterpoints=1, loc='upper right', frameon=False)
ax.add_artist(leg)

ax.set_title("UMAP + FoF graph")
ax.set_aspect('equal', 'box')

ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# … after computing X_emb, G, labels, z_vals, etc. …

# 1) compute sizes as a NumPy array
min_size, max_size = 20, 200
zmin, zmax = z_vals.min(), z_vals.max()
sizes = min_size + (z_vals - zmin) / (zmax - zmin) * (max_size - min_size)
# now ensure it really is an array
sizes = np.asarray(sizes)

# 2) identify outlier vs inlier nodes
counts = np.bincount(labels)
outlier_clusters = np.where(counts <= 1)[0]
is_outlier = np.isin(labels, outlier_clusters)

inlier_nodes  = np.where(~is_outlier)[0]
outlier_nodes = np.where(is_outlier)[0]

# 3) build a size lookup
size_map = {i: sizes[i] for i in range(len(sizes))}

# 4) plot
fig, ax = plt.subplots(figsize=(8,6))
pos = {i: X_emb[i] for i in range(len(X_emb))}

# draw edges
nx.draw_networkx_edges(G, pos, alpha=0.2, width=0.5, ax=ax)

# draw inliers
nx.draw_networkx_nodes(
    G, pos,
    nodelist=inlier_nodes.tolist(),
    node_size=1,
    node_color='k',
    alpha=0.6,
    ax=ax
)

# draw outliers
nx.draw_networkx_nodes(
    G, pos,
    nodelist=outlier_nodes.tolist(),
    node_size=5,
    node_color='r',
    alpha=0.9,
    ax=ax,
    label='Outliers'
)

ax.set_aspect('equal', 'box')
ax.legend(scatterpoints=1, fontsize='small', loc='upper right')
ax.set_title("UMAP + FoF: node size  redshift, outliers in red")
ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
import sys, os
project_root = os.path.abspath('..')
sys.path.insert(0, project_root)

from src.desiproc.build_matrix import build_matrix
import glob

out_dir = os.path.join(project_root, 'AssessingDesiData', 'data', 'processed')
night   = '20211130'
tile    = '5568'
bands   = ['B','R','Z']
wg, fp, iv, z, ze, ids, cat, petals = build_matrix(out_dir, night, tile, bands)

print("wave_grid:", wg.shape)
print("flux matrix:", fp.shape)
print("ivar matrix:", iv.shape)
print("z vector:", z.shape)
print("zerr vector:", ze.shape)
print("ids vector:", ids.shape)
print("cat matrix:", cat.shape)
print("petals matrix:", petals.shape)

In [ ]:
X, _ = fp, cat
y_true = _

In [ ]:
import numpy as np
from sklearn.neighbors import radius_neighbors_graph
from scipy.sparse.csgraph import connected_components
import networkx as nx
import matplotlib.pyplot as plt
from umap import UMAP

# 1) assume you have already done:
# from src.desiproc.build_matrix import build_matrix
# wg, fp, iv, z, ze, ids, cat, petals = build_matrix(...)

# 2) UMAP embedding on the flux matrix
reducer = UMAP(n_neighbors=50, min_dist=0.3, spread=3.0)
X_emb = reducer.fit_transform(fp)

# 3) Friend-of-Friend adjacency in UMAP space
radius = 0.3
adj = radius_neighbors_graph(X_emb, radius=radius, include_self=False)

# 4) Build NetworkX graph
rows, cols = adj.nonzero()
G = nx.Graph()
G.add_edges_from(zip(rows, cols))

# 5) Map redshift → marker size
min_size, max_size = 20, 200
zmin, zmax = z.min(), z.max()
sizes = min_size + (z - zmin) / (zmax - zmin) * (max_size - min_size)

# 6) Plot
fig, ax = plt.subplots(figsize=(8,6))
pos = {i: X_emb[i] for i in range(len(X_emb))}

nx.draw_networkx_edges(G, pos, alpha=0.2, width=0.5, ax=ax)
nx.draw_networkx_nodes(
    G, pos,
    nodelist=list(G.nodes()),
    node_size=10,#sizes[list(G.nodes())],
    node_color='k',
    alpha=0.8,
    ax=ax
)

ax.set_aspect('equal', 'box')
ax.set_title("UMAP + FoF graph (node size  redshift)")
ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
len(z)

# otro

In [ ]:
file = np.load('./data/processed/umap/umap_20211110_10256.npz', allow_pickle=True)
file.files

In [ ]:
X_emb, _ = file['embedding'], file['ids']
cat = file['categories']
outl = file['outlier_mask']

In [ ]:
cat = [c.decode('utf-8') if isinstance(c, (bytes, bytearray)) else str(c)
                  for c in cat]
cat

In [ ]:
cmap = sns.color_palette("mako", as_cmap=True)
cmap

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.neighbors import radius_neighbors_graph

# … cálculo de outliers, sizes y pos …

# 1) construye la matriz de adyacencia
adj = radius_neighbors_graph(X_emb, radius=0.45, include_self=False)

# 2) elimina conexiones de outliers
outliers = np.where(outl)[0]
adj = adj.tolil()
adj[outliers, :] = 0
adj[:, outliers] = 0
adj = adj.tocsr()

# 3) construye el grafo usando from_scipy_sparse_array
G = nx.convert_matrix.from_scipy_sparse_array(adj)
pos = {i: X_emb[i] for i in range(len(X_emb))}
# 4) dibuja
fig, ax = plt.subplots(figsize=(8, 6))
nx.draw_networkx_edges(G, pos, alpha=0.2, width=0.5, ax=ax)
nx.draw_networkx_nodes(G, pos, nodelist=list(G.nodes()), node_size=1, node_color='k', alpha=0.8, ax=ax,
                       label='Nodes')
nx.draw_networkx_nodes(G, pos, nodelist=outliers.tolist(), node_size=40, node_color='firebrick', alpha=0.7,
                       ax=ax, node_shape='o', label='Outliers', linewidths=1.5)

# ax.set_title("UMAP + FoF graph", fontweight='bold', fontsize=16)
ax.set_aspect('equal', 'box')
ax.axis("off")
plt.legend(scatterpoints=1, fontsize='small', loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.neighbors import radius_neighbors_graph
from mpl_toolkits.axes_grid1.inset_locator import zoomed_inset_axes, mark_inset

# … cálculo de X_emb, outl, sizes …

# 1) construye la matriz de adyacencia
adj = radius_neighbors_graph(X_emb, radius=0.45, include_self=False)

# 2) elimina conexiones de outliers
outliers = np.where(outl)[0]
adj = adj.tolil()
adj[outliers, :] = 0
adj[:, outliers] = 0
adj = adj.tocsr()

# 3) construye el grafo
G = nx.convert_matrix.from_scipy_sparse_array(adj)
pos = {i: X_emb[i] for i in range(len(X_emb))}

# 4) dibuja la figura principal
fig, ax = plt.subplots(figsize=(8, 6))
nx.draw_networkx_edges(G, pos, alpha=0.2, width=0.5, ax=ax)
nx.draw_networkx_nodes(G, pos,
                       nodelist=list(G.nodes()),
                       node_size=1, node_color='k', alpha=0.8,
                       ax=ax)
nx.draw_networkx_nodes(G, pos,
                       nodelist=outliers.tolist(),
                       node_size=20, node_color='firebrick', alpha=0.9,
                       ax=ax, node_shape='x', linewidths=1.5)

ax.set_aspect('equal', 'box')
ax.axis("off")
plt.legend(['Edges','Nodes','Outliers'], scatterpoints=1, fontsize='small', loc='upper right')

# ——— inset zoom ———
# elige un outlier para hacer zoom
i0 = outliers[0]
x0, y0 = X_emb[i0]

# crea ejes inset con un factor de zoom
axins = zoomed_inset_axes(ax, zoom=4, loc='lower left')  # zoom=4×, posición abajo-izq
# vuelve a dibujar en el inset
nx.draw_networkx_edges(G, pos, alpha=0.2, width=0.5, ax=axins)
nx.draw_networkx_nodes(G, pos,
                       nodelist=list(G.nodes()),
                       node_size=1, alpha=0.8, ax=axins)
nx.draw_networkx_nodes(G, pos,
                       nodelist=[i0],
                       node_size=50, node_color='firebrick',
                       node_shape='x', ax=axins)

# fija límites del inset alrededor del outlier
delta = 0.6
axins.set_xlim(x0 - delta, x0 + delta)
axins.set_ylim(y0 - delta, y0 + delta)
axins.set_aspect('equal', 'box')
axins.axis('off')

# dibuja líneas que conectan el inset con la región original
mark_inset(ax, axins, loc1=2, loc2=4, fc="none", ec="0.5")

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.neighbors import radius_neighbors_graph

# … cálculo de outliers, sizes y pos …

# 1) construye la matriz de adyacencia y aísla outliers
adj = radius_neighbors_graph(X_emb, radius=0.45, include_self=False).tolil()
adj[outliers, :] = 0
adj[:, outliers] = 0
adj = adj.tocsr()

# 2) construye el grafo
G = nx.convert_matrix.from_scipy_sparse_array(adj)

# 3) prepara el colormap para las categorías
cats = np.array(cat)                         # tu array de categorías, shape = (n_nodos,)
unique = np.unique(cats)                     # categorías únicas
cmap = plt.get_cmap('tab10', len(unique))    # usa 'tab10' o el que prefieras

# 4) dibuja
fig, ax = plt.subplots(figsize=(8, 6))

# 4a) aristas
nx.draw_networkx_edges(G, pos, alpha=0.2, width=0.5, ax=ax)

# 4b) nodos por categoría (excluyendo outliers)
for i, category in enumerate(unique):
    # selecciona nodos de esta categoría que NO sean outlier
    mask = (cats == category)
    nodes = np.nonzero(mask & ~outl)[0].tolist()
    nx.draw_networkx_nodes(
        G, pos,
        nodelist=nodes,
        node_size=10,
        node_color=[cmap(i)],
        alpha=0.8,
        ax=ax,
        label=f'{category}'
    )

# 4c) outliers en rojo
nx.draw_networkx_nodes(
    G, pos,
    nodelist=outliers.tolist(),
    node_size=10,
    node_color='red',
    alpha=0.9,
    ax=ax,
    label='Outliers'
)

ax.set_title("UMAP + FoF graph")
ax.set_aspect('equal', 'box')
ax.axis("off")
plt.legend(scatterpoints=1, fontsize='small', loc='upper right')
plt.tight_layout()
plt.show()


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.neighbors import radius_neighbors_graph
from matplotlib.patches import Circle

# … cálculo de outliers, sizes y pos …
# donde:
#   outliers: array de índices outlier
#   X_emb: array (n_nodos, 2) con las coordenadas UMAP
#   pos: dict {i: X_emb[i]}
#   cat: array de categorías shape=(n_nodos,)
#   outl: boolean mask same length as cat, True para outliers

# 1) construye la matriz de adyacencia y aísla outliers
adj = radius_neighbors_graph(X_emb, radius=0.45, include_self=False).tolil()
adj[outliers, :] = 0
adj[:, outliers] = 0
adj = adj.tocsr()

# 2) construye el grafo
G = nx.convert_matrix.from_scipy_sparse_array(adj)

# 3) prepara categorías
cats = np.array(cat)
unique = np.unique(cats)

# colormap para los círculos
cmap = plt.get_cmap('tab10', len(unique))

# 4) dibuja
fig, ax = plt.subplots(figsize=(8, 6))

# 4a) aristas


# 4d) círculos por categoría
for idx, category in enumerate(unique):
    # toma solo nodos de esta categoría y no outliers
    mask = (cats == category) & (~outl)
    if np.sum(mask) < 40:
        continue  # omite si hay muy pocos puntos
    
    coords = X_emb[mask]                      # (m,2) para esta categoría
    centroid = coords.mean(axis=0)            # centroide
    dists = np.linalg.norm(coords - centroid, axis=1)
    radius = np.percentile(dists, 40)         # cubre el 90% más denso
    
    circ = Circle(centroid, radius,
                  fill=True,
                #   edgecolor=cmap(idx),
                  facecolor=cmap(idx),
                  linewidth=2,
                  alpha=0.5,
                  label=f'Cat {category}')
    ax.add_patch(circ)
    
nx.draw_networkx_edges(G, pos, alpha=0.2, width=0.5, ax=ax)

# 4b) nodos normales en negro
nodes_normales = [i for i in G.nodes() if i not in outliers]
nx.draw_networkx_nodes(
    G, pos,
    nodelist=nodes_normales,
    node_size=1,
    node_color='k',
    alpha=0.8,
    ax=ax
)

# 4c) outliers en rojo
nx.draw_networkx_nodes(
    G, pos,
    nodelist=outliers.tolist(),
    node_size=5,
    node_color='red',
    alpha=1,
    ax=ax
)

ax.set_title("UMAP + FoF graph")
ax.set_aspect('equal', 'box')
ax.axis("off")
plt.legend(loc='upper right', fontsize='small', scatterpoints=1)
plt.tight_layout()
plt.show()